In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive testing and analysis of Therapeutic Area Sales Growth for Q1 and Q2 2023
# Purpose: To validate schema, perform sales aggregation, pivot, growth calculation, and ensure data quality for therapeutic area sales growth analysis
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script reads from the Unity Catalog table 'purgo_databricks.purgo_playground.hcp_overall_performance', validates schema and data types, aggregates sales by therapeutic area and quarter, pivots Q1/Q2 sales, calculates growth percentage, and sorts by top growth. It includes unit, integration, and data quality tests, and handles nulls and type mismatches gracefully.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DateType, DoubleType  
from pyspark.sql.utils import AnalysisException  

# ---------------------------
# 1. Schema Validation Tests
# ---------------------------

def validate_schema(df):
    """
    Validates the schema of the input DataFrame for required columns and types.

    Args:
        df (pyspark.sql.DataFrame): Input DataFrame

    Returns:
        bool: True if schema is valid, raises AssertionError otherwise
    """
    required_schema = {
        "Therapeutic_Area": StringType,
        "Sales_Amount": LongType,
        "Sales_Date": DateType
    }
    actual_schema = {field.name: type(field.dataType) for field in df.schema.fields}
    # Check for missing columns
    for col, dtype in required_schema.items():
        assert col in actual_schema, f"Missing required column: {col}"
        # Check for correct data type
        assert actual_schema[col] == dtype, f"Column {col} must be {dtype.__name__}, got {actual_schema[col].__name__}"
    return True

try:
    # Read the source table from Unity Catalog
    df_src = spark.table("purgo_databricks.purgo_playground.hcp_overall_performance")
except AnalysisException as e:
    # Handle missing table gracefully
    raise RuntimeError(f"Source table not found: {str(e)}")

# Validate schema
assert validate_schema(df_src), "Schema validation failed"

# ---------------------------
# 2. Data Quality Validation
# ---------------------------

def validate_data_quality(df):
    """
    Validates data quality for required fields: not null, correct types, and valid values.

    Args:
        df (pyspark.sql.DataFrame): Input DataFrame

    Returns:
        bool: True if data quality is valid, raises AssertionError otherwise
    """
    # Therapeutic_Area must not be null or empty
    null_area = df.filter(F.col("Therapeutic_Area").isNull() | (F.trim(F.col("Therapeutic_Area")) == ""))
    assert null_area.count() == 0, "Therapeutic_Area must not be null or empty"
    # Sales_Amount must not be null and must be bigint
    null_sales = df.filter(F.col("Sales_Amount").isNull())
    assert null_sales.count() == 0, "Sales_Amount must not be null"
    # Sales_Date must not be null and must be date
    null_date = df.filter(F.col("Sales_Date").isNull())
    assert null_date.count() == 0, "Sales_Date must not be null"
    # Sales_Date must be in 2023 Q1 or Q2
    df_date = df.withColumn("Year", F.year("Sales_Date")).withColumn("Quarter", F.quarter("Sales_Date"))
    invalid_date = df_date.filter(~((F.col("Year") == 2023) & (F.col("Quarter").isin([1,2]))))
    assert invalid_date.count() == 0, "Sales_Date must be in 2023 Q1 or Q2"
    return True

# Data quality validation
assert validate_data_quality(df_src), "Data quality validation failed"

# ---------------------------
# 3. Transformation: Extract Year/Quarter, Aggregate Sales
# ---------------------------

def aggregate_sales_by_area_quarter(df):
    """
    Aggregates total sales by Therapeutic_Area, Year, and Quarter.

    Args:
        df (pyspark.sql.DataFrame): Input DataFrame

    Returns:
        pyspark.sql.DataFrame: Aggregated DataFrame with columns Therapeutic_Area, Year, Quarter, Total_Sales
    """
    df_trans = df.withColumn("Year", F.year("Sales_Date")) \
                 .withColumn("Quarter", F.quarter("Sales_Date"))
    # Filter for 2023 Q1 and Q2 only
    df_filtered = df_trans.filter((F.col("Year") == 2023) & (F.col("Quarter").isin([1,2])))
    # Aggregate total sales
    df_agg = df_filtered.groupBy("Therapeutic_Area", "Year", "Quarter") \
                        .agg(F.sum("Sales_Amount").alias("Total_Sales"))
    return df_agg

df_agg = aggregate_sales_by_area_quarter(df_src)

# ---------------------------
# 4. Pivot Quarterly Sales for Comparison
# ---------------------------

def pivot_quarterly_sales(df):
    """
    Pivots the aggregated sales DataFrame to have Q1_Sales and Q2_Sales columns.

    Args:
        df (pyspark.sql.DataFrame): Aggregated DataFrame

    Returns:
        pyspark.sql.DataFrame: Pivoted DataFrame with Q1_Sales, Q2_Sales
    """
    df_pivot = df.groupBy("Therapeutic_Area", "Year") \
                 .pivot("Quarter", [1,2]) \
                 .agg(F.first("Total_Sales"))
    # Fill missing sales with 0
    df_pivot = df_pivot.withColumn("Q1_Sales", F.coalesce(F.col("1"), F.lit(0)).cast(LongType())) \
                       .withColumn("Q2_Sales", F.coalesce(F.col("2"), F.lit(0)).cast(LongType()))
    return df_pivot

df_pivot = pivot_quarterly_sales(df_agg)

# ---------------------------
# 5. Calculate Growth Percentage
# ---------------------------

def calculate_growth_percentage(df):
    """
    Calculates growth percentage from Q1 to Q2 sales.

    Args:
        df (pyspark.sql.DataFrame): Pivoted DataFrame

    Returns:
        pyspark.sql.DataFrame: DataFrame with Growth_Percentage column
    """
    df_growth = df.withColumn(
        "Growth_Percentage",
        F.when(F.col("Q1_Sales") != 0,
               ((F.col("Q2_Sales") - F.col("Q1_Sales")) / F.col("Q1_Sales") * 100).cast(DoubleType())
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    return df_growth

df_growth = calculate_growth_percentage(df_pivot)

# ---------------------------
# 6. Final Output Selection and Sorting
# ---------------------------

def select_and_sort_final(df):
    """
    Selects required columns and sorts by Growth_Percentage descending.

    Args:
        df (pyspark.sql.DataFrame): DataFrame with growth calculation

    Returns:
        pyspark.sql.DataFrame: Final output DataFrame
    """
    df_final = df.select(
        "Therapeutic_Area", "Year", "Q1_Sales", "Q2_Sales", "Growth_Percentage"
    ).orderBy(F.col("Growth_Percentage").desc_nulls_last())
    return df_final

df_final = select_and_sort_final(df_growth)

# ---------------------------
# 7. Unit Tests for Transformations
# ---------------------------

def test_aggregate_sales_by_area_quarter():
    """
    Unit test for aggregate_sales_by_area_quarter function.
    """
    # Prepare test data
    test_data = [
        ("Cardiology", 15000, "2023-01-15"),
        ("Oncology", 20000, "2023-04-10"),
        ("Neurology", 8000, "2023-02-20"),
        ("Diabetes", 10000, "2023-03-05"),
    ]
    schema = StructType([
        StructField("Therapeutic_Area", StringType(), True),
        StructField("Sales_Amount", LongType(), True),
        StructField("Sales_Date", DateType(), True)
    ])
    df_test = spark.createDataFrame(test_data, schema)
    df_agg_test = aggregate_sales_by_area_quarter(df_test)
    # Assert correct aggregation
    assert df_agg_test.count() == 4, "Aggregation count mismatch"
    # Assert sum values
    result = {row["Therapeutic_Area"]: row["Total_Sales"] for row in df_agg_test.collect()}
    assert result["Cardiology"] == 15000, "Cardiology sales mismatch"
    assert result["Oncology"] == 20000, "Oncology sales mismatch"
    assert result["Neurology"] == 8000, "Neurology sales mismatch"
    assert result["Diabetes"] == 10000, "Diabetes sales mismatch"

def test_pivot_quarterly_sales():
    """
    Unit test for pivot_quarterly_sales function.
    """
    test_data = [
        ("Cardiology", 2023, 1, 15000),
        ("Cardiology", 2023, 2, 30000),
        ("Oncology", 2023, 1, 20000),
        ("Oncology", 2023, 2, 35000),
    ]
    schema = StructType([
        StructField("Therapeutic_Area", StringType(), True),
        StructField("Year", IntegerType(), True),
        StructField("Quarter", IntegerType(), True),
        StructField("Total_Sales", LongType(), True)
    ])
    df_test = spark.createDataFrame(test_data, schema)
    df_pivot_test = pivot_quarterly_sales(df_test)
    # Assert columns
    assert "Q1_Sales" in df_pivot_test.columns and "Q2_Sales" in df_pivot_test.columns, "Pivot columns missing"
    # Assert values
    result = {row["Therapeutic_Area"]: (row["Q1_Sales"], row["Q2_Sales"]) for row in df_pivot_test.collect()}
    assert result["Cardiology"] == (15000, 30000), "Cardiology pivot mismatch"
    assert result["Oncology"] == (20000, 35000), "Oncology pivot mismatch"

def test_calculate_growth_percentage():
    """
    Unit test for calculate_growth_percentage function.
    """
    test_data = [
        ("Cardiology", 2023, 15000, 30000),
        ("Oncology", 2023, 20000, 35000),
        ("Diabetes", 2023, 10000, 9000),
        ("Immunology", 2023, 0, 20000),
    ]
    schema = StructType([
        StructField("Therapeutic_Area", StringType(), True),
        StructField("Year", IntegerType(), True),
        StructField("Q1_Sales", LongType(), True),
        StructField("Q2_Sales", LongType(), True)
    ])
    df_test = spark.createDataFrame(test_data, schema)
    df_growth_test = calculate_growth_percentage(df_test)
    result = {row["Therapeutic_Area"]: row["Growth_Percentage"] for row in df_growth_test.collect()}
    assert abs(result["Cardiology"] - 100.0) < 0.01, "Cardiology growth mismatch"
    assert abs(result["Oncology"] - 75.0) < 0.01, "Oncology growth mismatch"
    assert abs(result["Diabetes"] + 10.0) < 0.01, "Diabetes growth mismatch"
    assert result["Immunology"] is None, "Immunology growth should be null"

# Run unit tests
test_aggregate_sales_by_area_quarter()
test_pivot_quarterly_sales()
test_calculate_growth_percentage()

# ---------------------------
# 8. Integration Test: End-to-End Flow
# ---------------------------

def integration_test_end_to_end():
    """
    Integration test for the full sales growth analysis pipeline.
    """
    test_data = [
        ("Cardiology", 15000, "2023-01-15"),
        ("Cardiology", 30000, "2023-04-10"),
        ("Oncology", 20000, "2023-01-20"),
        ("Oncology", 35000, "2023-04-15"),
        ("Diabetes", 10000, "2023-03-05"),
        ("Diabetes", 9000, "2023-04-20"),
        ("Immunology", 0, "2023-01-10"),
        ("Immunology", 20000, "2023-05-18"),
    ]
    schema = StructType([
        StructField("Therapeutic_Area", StringType(), True),
        StructField("Sales_Amount", LongType(), True),
        StructField("Sales_Date", DateType(), True)
    ])
    df_test = spark.createDataFrame(test_data, schema)
    df_agg_test = aggregate_sales_by_area_quarter(df_test)
    df_pivot_test = pivot_quarterly_sales(df_agg_test)
    df_growth_test = calculate_growth_percentage(df_pivot_test)
    df_final_test = select_and_sort_final(df_growth_test)
    # Assert final output columns
    expected_cols = ["Therapeutic_Area", "Year", "Q1_Sales", "Q2_Sales", "Growth_Percentage"]
    assert df_final_test.columns == expected_cols, "Final output columns mismatch"
    # Assert sorting by growth
    rows = df_final_test.collect()
    assert rows[0]["Therapeutic_Area"] == "Cardiology", "Top growth area mismatch"
    assert rows[0]["Growth_Percentage"] >= rows[1]["Growth_Percentage"], "Sorting by growth failed"

integration_test_end_to_end()

# ---------------------------
# 9. Data Type Conversion and NULL Handling Tests
# ---------------------------

def test_data_type_conversion_and_nulls():
    """
    Tests data type conversions and NULL handling for complex scenarios.
    """
    test_data = [
        ("Cardiology", 15000, "2023-01-15"),
        ("Oncology", None, "2023-04-10"),  # NULL Sales_Amount
        ("Neurology", 8000, None),         # NULL Sales_Date
        ("Diabetes", "abc", "2023-03-05"), # Invalid Sales_Amount
        ("", 8000, "2023-04-20"),          # Empty Therapeutic_Area
    ]
    schema = StructType([
        StructField("Therapeutic_Area", StringType(), True),
        StructField("Sales_Amount", StringType(), True),  # Intentionally as string for type mismatch
        StructField("Sales_Date", StringType(), True)
    ])
    df_test = spark.createDataFrame(test_data, schema)
    # Attempt to cast Sales_Amount to LongType and Sales_Date to DateType
    df_cast = df_test.withColumn("Sales_Amount", F.col("Sales_Amount").cast(LongType())) \
                     .withColumn("Sales_Date", F.to_date("Sales_Date"))
    # Validate data quality
    try:
        validate_data_quality(df_cast)
        assert False, "Expected data quality validation to fail"
    except AssertionError as e:
        # Expected failure due to nulls and type mismatches
        pass

test_data_type_conversion_and_nulls()

# ---------------------------
# 10. Performance Test: Large Data Simulation
# ---------------------------

def performance_test_large_data():
    """
    Performance test for large data volume simulation.
    """
    import random  
    from datetime import datetime  
    # Generate 100,000 rows for 10 therapeutic areas, random sales in Q1/Q2 2023
    areas = [f"TA_{i}" for i in range(10)]
    data = []
    for i in range(100000):
        area = random.choice(areas)
        quarter = random.choice([1,2])
        year = 2023
        month = random.choice([1,2,3]) if quarter == 1 else random.choice([4,5,6])
        day = random.randint(1,28)
        sales_date = datetime(year, month, day)
        sales_amount = random.randint(1000, 50000)
        data.append((area, sales_amount, sales_date))
    schema = StructType([
        StructField("Therapeutic_Area", StringType(), True),
        StructField("Sales_Amount", LongType(), True),
        StructField("Sales_Date", DateType(), True)
    ])
    df_perf = spark.createDataFrame(data, schema)
    # Run full pipeline and time execution
    import time  
    start = time.time()
    df_agg_perf = aggregate_sales_by_area_quarter(df_perf)
    df_pivot_perf = pivot_quarterly_sales(df_agg_perf)
    df_growth_perf = calculate_growth_percentage(df_pivot_perf)
    df_final_perf = select_and_sort_final(df_growth_perf)
    duration = time.time() - start
    # Assert performance threshold (e.g., < 10 seconds)
    assert duration < 10, f"Performance test failed: {duration:.2f}s"

performance_test_large_data()

# ---------------------------
# 11. Delta Lake Operations Test (if applicable)
# ---------------------------

def test_delta_lake_operations():
    """
    Tests Delta Lake MERGE, UPDATE, DELETE operations on a test Delta table.
    """
    # Create test Delta table
    delta_table_name = "purgo_playground.test_therapeutic_area_sales"
    # Drop table if exists
    spark.sql(f"DROP TABLE IF EXISTS {delta_table_name}")
    # Create table with schema
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {delta_table_name} (
            Therapeutic_Area STRING NOT NULL,
            Year INT NOT NULL,
            Q1_Sales BIGINT NOT NULL,
            Q2_Sales BIGINT NOT NULL,
            Growth_Percentage DOUBLE,
            CONSTRAINT year_check CHECK (Year = 2023),
            CONSTRAINT area_not_null CHECK (Therapeutic_Area IS NOT NULL)
        ) USING DELTA
    """)
    # Insert test data
    df_insert = spark.createDataFrame([
        ("Cardiology", 2023, 15000, 30000, 100.0),
        ("Oncology", 2023, 20000, 35000, 75.0)
    ], ["Therapeutic_Area", "Year", "Q1_Sales", "Q2_Sales", "Growth_Percentage"])
    # Ensure column count matches table schema
    assert len(df_insert.columns) == 5, "Column count mismatch for Delta insert"
    df_insert.write.format("delta").mode("append").saveAsTable(delta_table_name)
    # MERGE: Upsert new row
    spark.sql(f"""
        MERGE INTO {delta_table_name} AS tgt
        USING (SELECT "Neurology" AS Therapeutic_Area, 2023 AS Year, 8000 AS Q1_Sales, 12000 AS Q2_Sales, 50.0 AS Growth_Percentage) AS src
        ON tgt.Therapeutic_Area = src.Therapeutic_Area AND tgt.Year = src.Year
        WHEN MATCHED THEN UPDATE SET Q1_Sales = src.Q1_Sales, Q2_Sales = src.Q2_Sales, Growth_Percentage = src.Growth_Percentage
        WHEN NOT MATCHED THEN INSERT (Therapeutic_Area, Year, Q1_Sales, Q2_Sales, Growth_Percentage) VALUES (src.Therapeutic_Area, src.Year, src.Q1_Sales, src.Q2_Sales, src.Growth_Percentage)
    """)
    # UPDATE: Change Q2_Sales for Cardiology
    spark.sql(f"""
        UPDATE {delta_table_name}
        SET Q2_Sales = 35000, Growth_Percentage = ((35000 - Q1_Sales) / Q1_Sales * 100)
        WHERE Therapeutic_Area = "Cardiology" AND Year = 2023
    """)
    # DELETE: Remove Oncology row
    spark.sql(f"""
        DELETE FROM {delta_table_name}
        WHERE Therapeutic_Area = "Oncology" AND Year = 2023
    """)
    # Validate final table
    df_delta = spark.table(delta_table_name)
    assert df_delta.count() == 2, "Delta table row count mismatch after DELETE"
    # Cleanup
    spark.sql(f"DROP TABLE IF EXISTS {delta_table_name}")

test_delta_lake_operations()

# ---------------------------
# 12. Window Function and Analytics Feature Test
# ---------------------------

def test_window_functions():
    """
    Tests window functions for ranking therapeutic areas by growth.
    """
    from pyspark.sql.window import Window  
    window_spec = Window.orderBy(F.col("Growth_Percentage").desc_nulls_last())
    df_ranked = df_final.withColumn("Growth_Rank", F.rank().over(window_spec))
    # Assert rank values
    ranks = [row["Growth_Rank"] for row in df_ranked.collect()]
    assert ranks == sorted(ranks), "Window function ranking failed"

test_window_functions()

# ---------------------------
# 13. Complex Type Validation (ARRAY, STRUCT, MAP)
# ---------------------------

def test_complex_types():
    """
    Tests creation and validation of complex types (ARRAY, STRUCT, MAP).
    """
    # ARRAY: List of sales per quarter
    df_array = df_final.withColumn("Sales_Array", F.array("Q1_Sales", "Q2_Sales"))
    # STRUCT: Combine sales and growth
    df_struct = df_final.withColumn("Sales_Struct", F.struct("Q1_Sales", "Q2_Sales", "Growth_Percentage"))
    # MAP: Map quarter to sales
    df_map = df_final.withColumn("Sales_Map", F.create_map(F.lit("Q1"), F.col("Q1_Sales"), F.lit("Q2"), F.col("Q2_Sales")))
    # Assert complex types
    row = df_array.first()
    assert isinstance(row["Sales_Array"], list), "ARRAY type validation failed"
    row = df_struct.first()
    assert hasattr(row["Sales_Struct"], "Q1_Sales"), "STRUCT type validation failed"
    row = df_map.first()
    assert "Q1" in row["Sales_Map"], "MAP type validation failed"

test_complex_types()

# ---------------------------
# 14. Final Output Display (for manual inspection)
# ---------------------------

# Display final results for manual inspection (commented out for automated test environments)
# df_final.show()

# spark.stop()  # Do not stop SparkSession in Databricks
